In [138]:
words = open('names.txt', 'r').read().splitlines()

In [139]:
# bigrams = 28 * 28 array with each element being an int repr freq of how many times 
# char at index i followed char at index i+1

In [140]:
import torch

In [141]:
N = torch.zeros((27, 27), dtype=torch.int32)

In [142]:
charToInd = dict()
indToChar = dict()

allChars = sorted(set(''.join(words)))
allChars = ['.'] + allChars

index = 0
for char in allChars:
    print(char, index)
    charToInd[char] = index
    indToChar[index] = char
    index += 1



. 0
a 1
b 2
c 3
d 4
e 5
f 6
g 7
h 8
i 9
j 10
k 11
l 12
m 13
n 14
o 15
p 16
q 17
r 18
s 19
t 20
u 21
v 22
w 23
x 24
y 25
z 26


In [143]:
for word in words:
    N[0, charToInd[word[0]]] += 1
    N[charToInd[word[-1]], 0] += 1
    for char, nextChar in zip(word, word[1:]):
        idx1, idx2 = charToInd[char], charToInd[nextChar]
        N[idx1, idx2] += 1

In [144]:
print(N)

tensor([[   0, 4410, 1306, 1542, 1690, 1531,  417,  669,  874,  591, 2422, 2963,
         1572, 2538, 1146,  394,  515,   92, 1639, 2055, 1308,   78,  376,  307,
          134,  535,  929],
        [6640,  556,  541,  470, 1042,  692,  134,  168, 2332, 1650,  175,  568,
         2528, 1634, 5438,   63,   82,   60, 3264, 1118,  687,  381,  834,  161,
          182, 2050,  435],
        [ 114,  321,   38,    1,   65,  655,    0,    0,   41,  217,    1,    0,
          103,    0,    4,  105,    0,    0,  842,    8,    2,   45,    0,    0,
            0,   83,    0],
        [  97,  815,    0,   42,    1,  551,    0,    2,  664,  271,    3,  316,
          116,    0,    0,  380,    1,   11,   76,    5,   35,   35,    0,    0,
            3,  104,    4],
        [ 516, 1303,    1,    3,  149, 1283,    5,   25,  118,  674,    9,    3,
           60,   30,   31,  378,    0,    1,  424,   29,    4,   92,   17,   23,
            0,  317,    1],
        [3983,  679,  121,  153,  384, 1271,   82,

In [145]:
sumN = torch.sum(N, 1, keepdim=True)
sumN
prob = N / sumN
# print(torch.sum(M, 1))

In [146]:
# test1 = torch.randint(1,3, (2,2))
# print(test1)
# sumTest = torch.sum(test1, dim=1)
# print(sumTest)
# print(test1 / sumTest)

In [147]:
index = 0
word = ''
while True:
    distribution = prob[index]
    next_index = torch.multinomial(distribution, num_samples=1, replacement=True).item()

    if next_index == 0:
        break
    word += indToChar[next_index]
    index = next_index
print(word)

a


In [148]:
# loss function calculation
word = 'vedanta'
# what is our models probability to predict anna
ix1 = 0
prob_word = 1
for char in word:
    ix2 = charToInd[char]
    prob_word *= prob[ix1, ix2]
    ix1 = ix2
prob_word *= prob[ix1, 0]
print(prob_word)

# that is a really small number, let us do log and addition instead
log_liklihood = 0
for char in word:
    ix2 = charToInd[char]
    log_liklihood += torch.log(prob[ix1, ix2])
    ix1 = ix2
log_liklihood += torch.log(prob[ix1, 0])
print(-log_liklihood/len(word))



tensor(1.6189e-09)
tensor(2.7859)


Now moving onto building the same bigram but from NN now

In [ ]:
from torch.nn import functional as F
from torch import tensor

# forward pass
W = torch.randn(27,27)
# F.one_hot(torch.tensor([400,2,3]), num_classes=401)
# W

In [157]:
inputs, targets = [], []
for word in words:
    complete_word = '.'+word+'.'
    for char_input, char_target in zip(complete_word, complete_word[1:]):
        inputs.append(charToInd[char_input])
        targets.append(charToInd[char_target])

    break

In [199]:
encoded_input = F.one_hot(tensor(inputs), 27).float()
W = torch.randn((27,27))
logits = encoded_input @ W

# softmax
counts = logits.exp()
probs = counts / counts.sum(1, keepdim=True)


In [ ]:
probs.shape

for prob in probs:
    

torch.Size([5, 27])

In [196]:
counts = logits.exp()
probs = counts / torch.sum(counts, dim=1, keepdim=True)

In [197]:
probs

tensor([[0.0501, 0.1035, 0.0358, 0.0225, 0.0434, 0.0622, 0.0141, 0.0218, 0.0106,
         0.0635, 0.0841, 0.0269, 0.0577, 0.0047, 0.0094, 0.0298, 0.0010, 0.0068,
         0.0109, 0.0248, 0.0338, 0.0938, 0.0579, 0.0754, 0.0338, 0.0120, 0.0095],
        [0.0222, 0.0449, 0.0532, 0.0013, 0.0380, 0.0123, 0.0323, 0.0677, 0.0503,
         0.0223, 0.0668, 0.0109, 0.0168, 0.0894, 0.1051, 0.0353, 0.0034, 0.0359,
         0.0071, 0.0039, 0.0178, 0.0206, 0.0259, 0.0594, 0.1153, 0.0283, 0.0136],
        [0.0176, 0.0147, 0.0044, 0.0044, 0.0259, 0.0017, 0.0442, 0.0047, 0.0101,
         0.0338, 0.0769, 0.2290, 0.0898, 0.0100, 0.0203, 0.0086, 0.0924, 0.0154,
         0.0014, 0.0296, 0.0756, 0.0484, 0.0486, 0.0062, 0.0108, 0.0176, 0.0577],
        [0.0176, 0.0147, 0.0044, 0.0044, 0.0259, 0.0017, 0.0442, 0.0047, 0.0101,
         0.0338, 0.0769, 0.2290, 0.0898, 0.0100, 0.0203, 0.0086, 0.0924, 0.0154,
         0.0014, 0.0296, 0.0756, 0.0484, 0.0486, 0.0062, 0.0108, 0.0176, 0.0577],
        [0.0123, 0.0103,

In [ ]:
# torch.sum(probs, dim=1, keepdim=True)

tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])